In [1]:
import re
from datetime import datetime
import pandas as pd
from dateutil import parser as dateparser

LINE_RE = re.compile(r"^\[(.+?)\] ([^:]+): (.*)$")

def parse_export(path):
    messages = []
    current = None
    with open(path, encoding="utf-8") as f:
        for line in f:
            m = LINE_RE.match(line)
            if m:
                if current:
                    messages.append(current)
                ts, sender, body = m.groups()
                current = {
                    "timestamp": dateparser.parse(ts, fuzzy=True),
                    "sender": sender.strip(),
                    "body": body.rstrip("\n"),
                }
            elif current:
                current["body"] += "\n" + line.rstrip("\n")
    if current:
        messages.append(current)
    df = pd.DataFrame(messages)
    df["is_media"] = df["body"].str.contains("<Media omitted>", na=False)
    df["is_system"] = df["sender"] == "System"
    return df

In [2]:
df = parse_export("sample_chat.txt")
print(df.head())

            timestamp      sender                                    body  \
0 2026-03-15 14:23:01    John Doe                   Welcome to the group!   
1 2026-03-15 14:24:15  Jane Smith                         <Media omitted>   
2 2026-03-15 14:25:30    John Doe  When is the next meeting we agreed on?   
3 2026-03-15 14:26:02      System               John Doe added Bob Wilson   
4 2026-03-15 14:27:10  Bob Wilson       Hi everyone, thanks for adding me   

   is_media  is_system  
0     False      False  
1      True      False  
2     False      False  
3     False       True  
4     False      False  


First we cover the easy anaytics:
1. Hour frequency
2. User frequency
3. Daily and weekly frequencies

In [3]:
# create the summary
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

_vader = SentimentIntensityAnalyzer()

def sentiment_analysis(df):
    real_mask = ~df["is_system"]
    df.loc[real_mask, "sentiment"] = df.loc[real_mask, "body"].apply(
        lambda text: _vader.polarity_scores(text)["compound"]
    )

def compute_summary(df):
    real = df[~df['is_system']]
    return {
        "total_messages": int(len(real)),
        "unique_senders": int(real["sender"].nunique()),
        "media_message_count": int(real['is_media'].sum()),
        "date_range": {
            "first": real["timestamp"].min().date(),
            "last": real["timestamp"].max().date()
        },       
        "text_message_count": int((~real['is_media']).sum())
    }

def compute_peak_hours(df):
    counts = df[~df["is_system"]]["timestamp"].dt.hour.value_counts().reindex(range(24), fill_value=0)
    return [{"hour": int(h), "count": int(c)} for h, c in counts.items()]

def compute_per_user(df):
    sentiment_analysis(df)
    real = df[~df["is_system"]]
    return (
        real
        .groupby("sender")
        .agg(
            messages=("body", "count"),
            media=("is_media", "sum"),
            avg_sentiment=("sentiment", "mean")
        )
        .sort_values("messages", ascending=False)
    )

def compute_sentiment(df):
    sentiment_analysis(df)
    real = df[~df["is_system"]]
    compound = real["sentiment"]
    bucket = pd.cut(
        compound,
        bins=[-1.01, -0.05, 0.05, 1.01],
        labels=["negative", "neutral", "positive"],
    )
    fractions = bucket.value_counts(normalize=True)
    return {label: float(fractions.get(label, 0.0)) for label in ["positive", "neutral", "negative"]}

def compute_top_influencers(df):
    return df[~df["is_system"]]["sender"].value_counts().head(5).index.tolist()

def compute_frequency(df):
    real = df[~df["is_system"]]
    iso_date = real["timestamp"].dt.isocalendar()
    real = real.copy()
    real["year_week"] = iso_date["year"].astype(str) + "-W" + iso_date["week"].astype(str)

    daily = real.groupby(real["timestamp"].dt.date).size().rename("count")
    weekly = real.groupby(real["year_week"]).size().rename("count")

    return {
        "daily": daily,
        "weekly": weekly,
    }

print(compute_frequency(df))
print(df.groupby(df["sender"]).size().sort_values(ascending=False))
print(df.groupby(df["timestamp"].dt.date).size())
print(df.groupby(df["timestamp"].dt.isocalendar().week).size())

summary = compute_summary(df)
print(summary["total_messages"])
print(summary["text_message_count"])
print(summary["unique_senders"])
print(summary["media_message_count"])
print(summary["date_range"]["first"])
print(summary["date_range"]["last"])

print(compute_per_user(df))
print(compute_sentiment(df))


{'daily': timestamp
2026-03-15    57
2026-03-16    30
Name: count, dtype: int64, 'weekly': year_week
2026-W11    57
2026-W12    30
Name: count, dtype: int64}
sender
Jane Smith      24
John Doe        19
Mike Brown      17
Bob Wilson      15
Sarah Connor    12
System           9
dtype: int64
timestamp
2026-03-15    64
2026-03-16    32
dtype: int64
week
11    64
12    32
dtype: int64
87
86
5
1
2026-03-15
2026-03-16
              messages  media  avg_sentiment
sender                                      
Jane Smith          24      1       0.080267
John Doe            19      0       0.232389
Mike Brown          17      0       0.098971
Bob Wilson          15      0       0.167580
Sarah Connor        12      0       0.203358
{'positive': 0.3793103448275862, 'neutral': 0.5747126436781609, 'negative': 0.04597701149425287}
